# Аналитика по листу «Заказы белые ТЕСТ»

Ноутбук читает данные из Google Sheets таблицы `Расчет поставки Китай_по обороту`, лист `Заказы белые ТЕСТ`, и формирует аналитический датафрейм `df_orders_white_test` по контролируемому списку колонок.

In [72]:
from datetime import datetime
from pathlib import Path
import re
import sys

import pandas as pd
from IPython.display import display

# Делаем импорты проекта устойчивыми при запуске из корня проекта или из папки polygon.
project_root = Path.cwd().resolve()
if project_root.name == "polygon":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

project_root

WindowsPath('C:/Users/123/Desktop/start_vector')

In [73]:
from src_oop.core.my_gspread import GoogleTabs
from src_oop.jobs.calculation_of_purchases_china.config import delivery_calculation_china
from src_oop.core.utils_general import clean_currency_value

TABLE_TITLE = delivery_calculation_china["title"]
SHEET_TITLE = delivery_calculation_china["white_orders_sheet"]
TEST_SHEET_TITLE = delivery_calculation_china["test_sheet"]

# Загружаем данные из Google Sheets в DataFrame.
REQUIRED_COLUMNS = [
    "wild",
    "Модель",
    "Номер заказа 1С",
    "№ проформы в документах и 1С",
    "КОМПАНИЯ",
    "Поставщик",
    "Закупщик",
    "Статус",
    "Кол-во к заказу",
    "ФИН Статус",
    "Оплата аванса",
    "Дата аванса годовому плану",
    "Дата аванса план",
    "Дата для платеж календаря",
    "Дата аванса факт",
    "% Аванса",
    "Сумма аванса, RMB 🤖",
    "Оплата Баланса1",
    "Дата баланса1 план",
    "Дата баланса1 для платеж календаря",
    "Дата баланса факт",
    "% баланса",
    "Сумма баланса, RMB 🤖",
    "Оплата баланса",
    "Дата баланса план",
    "Дата баланса для платеж календаря",
    "Дата баланса факт1",
    "% баланса 🤖",
    "Сумма баланса, RMB 2 🤖"
]


# === Колонки, которые должны быть в итоговом DataFrame, сгруппированы по смыслу для удобства. ===
AVANS_COLS = ["Оплата аванса",
    "Дата аванса годовому плану",
    "Дата аванса план",
    "Дата для платеж календаря",
    "Дата аванса факт",
    "% Аванса",
    "Сумма аванса, RMB 🤖"]
BALANCE_ONE_COLS = ["Оплата Баланса1",
    "Дата баланса1 план",
    "Дата баланса1 для платеж календаря",
    "Дата баланса факт",
    "% баланса",
    "Сумма баланса, RMB 🤖"
]
BALANCE_TWO_COLS = ["Оплата баланса",  
    "Дата баланса план",
    "Дата баланса для платеж календаря",
    "Дата баланса факт1",
    "% баланса 🤖",
    "Сумма баланса, RMB 2 🤖"
]
# ===


# Статусы, по которым заказы считаются отмененными и не должны попадать в итоговый DataFrame.
CANCELED_STATUSES = ["Отмена", "в Планах", " "]

DIGIT_COLS = [
    "Кол-во к заказу",
    "Сумма аванса, RMB 🤖",
    "Сумма баланса, RMB 🤖",
    "Сумма баланса, RMB 2 🤖"
]

In [74]:
def normalize_column_name(column_name: object) -> str:
    """Приводит название колонки к стабильному виду для поиска по имени."""
    normalized = str(column_name).replace("\t", " ").replace("\n", " ").replace("\r", " ")
    return re.sub(r"\s+", " ", normalized).strip()


def make_unique_column_names(column_names: list[str]) -> list[str]:
    """Добавляет номер к повторяющимся заголовкам, чтобы к ним можно было обращаться по имени."""
    seen_columns: dict[str, int] = {}
    unique_columns = []

    for column_name in column_names:
        seen_columns[column_name] = seen_columns.get(column_name, 0) + 1
        if seen_columns[column_name] == 1:
            unique_columns.append(column_name)
            continue

        unique_columns.append(f"{column_name} {seen_columns[column_name]}")

    return unique_columns


def build_dataframe_from_sheet_values(values: list[list[str]]) -> pd.DataFrame:
    """Собирает DataFrame: заголовки в 4-й строке, данные начинаются с 5-й."""
    if len(values) < 4:
        raise ValueError("В листе меньше 4 строк, строка заголовков не найдена.")

    headers = make_unique_column_names([normalize_column_name(header) for header in values[3]])
    rows = values[4:]
    df = pd.DataFrame(rows, columns=headers)
    for col in DIGIT_COLS:
        df[col] = df[col].apply(clean_currency_value)


    return df

def validate_required_columns(df: pd.DataFrame, required_columns: list[str]) -> None:
    """Проверяет, что все обязательные колонки доступны после нормализации."""
    missing_columns = [column for column in required_columns if column not in df.columns]
    if missing_columns:
        available_columns = df.columns.tolist()
        raise ValueError(
            "Не найдены обязательные колонки: "
            f"{missing_columns}. Доступные колонки после нормализации: {available_columns}"
        )

In [75]:
# Подключаемся к Google Sheets через существующий клиент проекта.
google_sheet = GoogleTabs(table_title=TABLE_TITLE, sheet_title=SHEET_TITLE)
sheet_values = google_sheet.sheet_title.get_all_values()

df_orders_white_test_raw = build_dataframe_from_sheet_values(sheet_values)

print(f"Таблица: {TABLE_TITLE}")
print(f"Лист: {SHEET_TITLE}")
print(f"Исходный размер DataFrame: {df_orders_white_test_raw.shape}")

✅ Успешное подключение к Расчет поставки Китай_по обороту -> Заказы белые ТЕСТ
Таблица: Расчет поставки Китай_по обороту
Лист: Заказы белые ТЕСТ
Исходный размер DataFrame: (1787, 201)


In [76]:
validate_required_columns(df_orders_white_test_raw, REQUIRED_COLUMNS)

# Итоговый аналитический DataFrame: только нужные колонки в заданном порядке.
df_orders_white_test = df_orders_white_test_raw.loc[:, REQUIRED_COLUMNS].copy()
# Фильтруем строки с отмененными заказами по статусу.
df_orders_white_test = df_orders_white_test[~df_orders_white_test["Статус"].isin(CANCELED_STATUSES)]

display(df_orders_white_test.head())
df_orders_white_test.shape

,wild,Модель,Номер заказа 1С,№ проформы в документах и 1С,КОМПАНИЯ,Поставщик,Закупщик,Статус,Кол-во к заказу,ФИН Статус,...,Дата баланса1 для платеж календаря,Дата баланса факт,% баланса,"Сумма баланса, RMB 🤖",Оплата баланса,Дата баланса план,Дата баланса для платеж календаря,Дата баланса факт1,% баланса 🤖,"Сумма баланса, RMB 2 🤖"
0,wild354,Набор кухон.принадлежностей 19в1 черный,СТУТ-000058,СТУТ-000058,СТАРТ,"NINGBO GENERAL UNION CO., LTD",Руслан,прибыло,9408.0,Оплачен полностью,...,,15.01.26,,0.0,оплачено,,,,90%,479666.88
1,wild355,Набор кухон.принадлежностей 21 в1 серый,СТУТ-000058,СТУТ-000058,СТАРТ,"NINGBO GENERAL UNION CO., LTD",Руслан,прибыло,10000.0,Оплачен полностью,...,,15.01.26,,0.0,оплачено,,,,90%,509850.00
2,wild352,Набор кухон.принадлежностей 19в1 белый силикон,СТУТ-000058,СТУТ-000058,СТАРТ,"NINGBO GENERAL UNION CO., LTD",Руслан,прибыло,8560.0,Оплачен полностью,...,,15.01.26,,0.0,оплачено,,,,90%,436431.60
3,wild354,Набор кухон.принадлежностей 19в1 черный,СТУТ-000058,СТУТ-000058,СТАРТ,"NINGBO GENERAL UNION CO., LTD",Руслан,прибыло,5592.0,Оплачен полностью,...,,15.01.26,,0.0,оплачено,,,,90%,285108.12
4,wild352,Набор кухон.принадлежностей 19в1 белый силикон,СТУТ-000058,СТУТ-000058,СТАРТ,"NINGBO GENERAL UNION CO., LTD",Руслан,прибыло,5592.0,Оплачен полностью,...,,15.01.26,,0.0,оплачено,,,,90%,285108.12


(1787, 29)

In [77]:
ORDER_ID_COLUMNS = [
    column
    for column in REQUIRED_COLUMNS
    if column not in AVANS_COLS + BALANCE_ONE_COLS + BALANCE_TWO_COLS
]
var = ["Статус_по_этапу", "Дата_аванса_по_годовому_плану", "Дата_план", "Дата_платеж_календарь", "Дата_факт", "%_оплаты", "Сумма_оплаты"]

avans = {
            "Оплата аванса": "Оплата аванса",
            "Дата аванса годовому плану": "Дата аванса годовому плану",
            "Дата аванса план": "Дата аванса план",
            "Дата для платеж календаря": "Дата для платеж календаря",
            "Дата аванса факт": "Дата аванса факт",
            "% Аванса": "% Аванса",
}

PAYMENT_CONFIGS = [
    {
        "Номер этапа платежа": 1,
        "Этап платежа": "Аванс",
        "Источник данных": "Колонки авансового платежа",
        "columns": {
            "Статус_по_этапу": "Оплата аванса",
            "Дата_аванса_по_годовому_плану": "Дата аванса годовому плану",
            "Дата_план": "Дата аванса план",
            "Дата_платеж_календарь": "Дата для платеж календаря",
            "Дата_факт": "Дата аванса факт",
            "%_оплаты": "% Аванса",
            "Сумма_оплаты": "Сумма аванса, RMB 🤖",
        },
    },
    {
        "Номер этапа платежа": 2,
        "Этап платежа": "Баланс 1",
        "Источник данных": "Колонки первого балансового платежа",
        "columns": {
            "Статус_по_этапу": "Оплата Баланса1",
            "Дата_аванса_по_годовому_плану": "Дата аванса годовому плану",
            "Дата_план": "Дата баланса1 план",
            "Дата_платеж_календарь": "Дата баланса1 для платеж календаря",
            "Дата_факт": "Дата баланса факт",
            "%_оплаты": "% баланса",
            "Сумма_оплаты": "Сумма баланса, RMB 🤖",
        },
    },
    {
        "Номер этапа платежа": 3,
        "Этап платежа": "Баланс 2",
        "Источник данных": "Колонки второго балансового платежа",
        "columns": {
            "Статус_по_этапу": "Оплата баланса",
            "Дата_аванса_по_годовому_плану": "Дата аванса годовому плану",
            "Дата_план": "Дата баланса план",
            "Дата_платеж_календарь": "Дата баланса для платеж календаря",
            "Дата_факт": "Дата баланса факт1",
            "%_оплаты": "% баланса 🤖",
            "Сумма_оплаты": "Сумма баланса, RMB 2 🤖",
        },
    },
]


def build_balance_payment_dataframe(
    df: pd.DataFrame,
    base_columns: list[str],
    payment_config: dict[str, object],
) -> pd.DataFrame:
    """Формирует строки одного этапа балансового платежа."""
    payment_columns = payment_config["columns"]
    validate_required_columns(df, base_columns + list(payment_columns.values()))

    balance_df = df.loc[:, base_columns].copy()
    balance_df["_Порядок исходной строки"] = range(len(df))
    balance_df["Этап платежа"] = payment_config["Этап платежа"]
    balance_df["Номер этапа платежа"] = payment_config["Номер этапа платежа"]

    for target_column, source_column in payment_columns.items():
        balance_df[target_column] = df[source_column].to_numpy()

    return balance_df


balance_payment_frames = [
    build_balance_payment_dataframe(
        df=df_orders_white_test,
        base_columns=ORDER_ID_COLUMNS,
        payment_config=payment_config,
    )
    for payment_config in PAYMENT_CONFIGS
]

df_orders_white_balance = pd.concat(balance_payment_frames, ignore_index=True)

# Исключаем только строки без суммы: пустая сумма не является платежом, а 0 может быть значимым значением.
df_orders_white_balance = df_orders_white_balance[
    df_orders_white_balance["Статус_по_этапу"].notna()
].copy()

df_orders_white_balance = df_orders_white_balance.sort_values(
    by=["Номер заказа 1С", "_Порядок исходной строки", "Номер этапа платежа"],
    kind="stable",
).drop(columns="_Порядок исходной строки").reset_index(drop=True)

paid_mask = df_orders_white_balance["Статус_по_этапу"].eq("оплачено")
payment_amount = df_orders_white_balance["Сумма_оплаты"].fillna(0)

df_orders_white_balance["Оплачено"] = payment_amount.where(paid_mask, 0)
df_orders_white_balance["Не_оплачено"] = payment_amount.where(~paid_mask, 0)


In [78]:
print(f"Размер df_orders_white_balance: {df_orders_white_balance.shape}")

display(df_orders_white_balance.head())

print("Количество строк по этапам платежа:")
display(df_orders_white_balance["Этап платежа"].value_counts(dropna=False))

print("Количество пропусков по колонкам:")
display(df_orders_white_balance.isna().sum())


Размер df_orders_white_balance: (5361, 21)


,wild,Модель,Номер заказа 1С,№ проформы в документах и 1С,КОМПАНИЯ,Поставщик,Закупщик,Статус,Кол-во к заказу,ФИН Статус,...,Номер этапа платежа,Статус_по_этапу,Дата_аванса_по_годовому_плану,Дата_план,Дата_платеж_календарь,Дата_факт,%_оплаты,Сумма_оплаты,Оплачено,Не_оплачено
0,wild172,Диспенсер кухонный 6л 6 отсеков 6 кнопка белый...,,B26GD04053-1 wild172,ВЕКТОР,"NINGBO GENERAL UNION CO., LTD",Руслан,в производстве,3348.0,Оплачен полностью,...,1,оплачено,,,,23.10.25,10%,10881.0,10881.0,0.0
1,wild172,Диспенсер кухонный 6л 6 отсеков 6 кнопка белый...,,B26GD04053-1 wild172,ВЕКТОР,"NINGBO GENERAL UNION CO., LTD",Руслан,в производстве,3348.0,Оплачен полностью,...,2,оплачено,,,,28.11.25,,0.0,0.0,0.0
2,wild172,Диспенсер кухонный 6л 6 отсеков 6 кнопка белый...,,B26GD04053-1 wild172,ВЕКТОР,"NINGBO GENERAL UNION CO., LTD",Руслан,в производстве,3348.0,Оплачен полностью,...,3,оплачено,,,,,90%,97929.0,97929.0,0.0
3,wild172,Диспенсер кухонный 6л 6 отсеков 6 кнопка белый...,,B26GD04053-2 wild172https://drive.google.com/d...,ВЕКТОР,"NINGBO GENERAL UNION CO., LTD",Руслан,в производстве,3348.0,Оплачен полностью,...,1,оплачено,,,,23.10.25,10%,10881.0,10881.0,0.0
4,wild172,Диспенсер кухонный 6л 6 отсеков 6 кнопка белый...,,B26GD04053-2 wild172https://drive.google.com/d...,ВЕКТОР,"NINGBO GENERAL UNION CO., LTD",Руслан,в производстве,3348.0,Оплачен полностью,...,2,оплачено,,,,28.11.25,,0.0,0.0,0.0


Количество строк по этапам платежа:


Этап платежа
Аванс       1787
Баланс 1    1787
Баланс 2    1787
Name: count, dtype: int64

Количество пропусков по колонкам:


wild                             0
Модель                           0
Номер заказа 1С                  0
№ проформы в документах и 1С     0
КОМПАНИЯ                         0
Поставщик                        0
Закупщик                         0
Статус                           0
Кол-во к заказу                  0
ФИН Статус                       0
Этап платежа                     0
Номер этапа платежа              0
Статус_по_этапу                  0
Дата_аванса_по_годовому_плану    0
Дата_план                        0
Дата_платеж_календарь            0
Дата_факт                        0
%_оплаты                         0
Сумма_оплаты                     0
Оплачено                         0
Не_оплачено                      0
dtype: int64

In [79]:
df_orders_white_balance_to_upload = df_orders_white_balance.copy()
print(f"Размер датафрейма для выгрузки: {df_orders_white_balance_to_upload.shape}")
display(df_orders_white_balance_to_upload.head())

if df_orders_white_balance_to_upload.empty:
    print("df_orders_white_balance_to_upload пустой, выгрузка в Google Sheets не выполнялась.")
else:
    google_sheet_to_upload = GoogleTabs(table_title=TABLE_TITLE, sheet_title=TEST_SHEET_TITLE)
    google_sheet_to_upload.set_df_to_google(df_orders_white_balance_to_upload)
    print(f"df_orders_white_balance выгружен на лист: {TEST_SHEET_TITLE}")

Размер датафрейма для выгрузки: (5361, 21)


,wild,Модель,Номер заказа 1С,№ проформы в документах и 1С,КОМПАНИЯ,Поставщик,Закупщик,Статус,Кол-во к заказу,ФИН Статус,...,Номер этапа платежа,Статус_по_этапу,Дата_аванса_по_годовому_плану,Дата_план,Дата_платеж_календарь,Дата_факт,%_оплаты,Сумма_оплаты,Оплачено,Не_оплачено
0,wild172,Диспенсер кухонный 6л 6 отсеков 6 кнопка белый...,,B26GD04053-1 wild172,ВЕКТОР,"NINGBO GENERAL UNION CO., LTD",Руслан,в производстве,3348.0,Оплачен полностью,...,1,оплачено,,,,23.10.25,10%,10881.0,10881.0,0.0
1,wild172,Диспенсер кухонный 6л 6 отсеков 6 кнопка белый...,,B26GD04053-1 wild172,ВЕКТОР,"NINGBO GENERAL UNION CO., LTD",Руслан,в производстве,3348.0,Оплачен полностью,...,2,оплачено,,,,28.11.25,,0.0,0.0,0.0
2,wild172,Диспенсер кухонный 6л 6 отсеков 6 кнопка белый...,,B26GD04053-1 wild172,ВЕКТОР,"NINGBO GENERAL UNION CO., LTD",Руслан,в производстве,3348.0,Оплачен полностью,...,3,оплачено,,,,,90%,97929.0,97929.0,0.0
3,wild172,Диспенсер кухонный 6л 6 отсеков 6 кнопка белый...,,B26GD04053-2 wild172https://drive.google.com/d...,ВЕКТОР,"NINGBO GENERAL UNION CO., LTD",Руслан,в производстве,3348.0,Оплачен полностью,...,1,оплачено,,,,23.10.25,10%,10881.0,10881.0,0.0
4,wild172,Диспенсер кухонный 6л 6 отсеков 6 кнопка белый...,,B26GD04053-2 wild172https://drive.google.com/d...,ВЕКТОР,"NINGBO GENERAL UNION CO., LTD",Руслан,в производстве,3348.0,Оплачен полностью,...,2,оплачено,,,,28.11.25,,0.0,0.0,0.0


✅ Успешное подключение к Расчет поставки Китай_по обороту -> Тестовая_выгрузка
✅ Успешное подключение к Расчет поставки Китай_по обороту -> Тестовая_выгрузка
Таблица полностью обновлена
df_orders_white_balance выгружен на лист: Тестовая_выгрузка
